<a href="https://colab.research.google.com/github/itsumar1923/Brain-Tumor-Classification/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
torch.cuda.is_available()


True

In [2]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True" # Added to mitigate memory fragmentation

In [3]:
!pip install -U pip
!pip install unsloth
!pip install pypdf datasets trl accelerate peft bitsandbytes transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 52.7 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 175.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 66.6 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 15.2 MB/s  0:00:23
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 153.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 38.6 MB/s  0:00:09
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 110.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━

In [1]:
from google.colab import files
files.upload()


Saving Resume_UFS.pdf to Resume_UFS.pdf


{'Resume_UFS.pdf': b'%PDF-1.5\n%\xbf\xf7\xa2\xfe\n3 0 obj\n<< /Linearized 1 /L 101883 /H [ 3449 141 ] /O 8 /E 101288 /N 1 /T 101599 >>\nendobj\n                                                                                                             \n4 0 obj\n<< /Type /XRef /Length 83 /Filter /FlateDecode /DecodeParms << /Columns 5 /Predictor 12 >> /W [ 1 3 1 ] /Index [ 3 48 ] /Info 1 0 R /Root 5 0 R /Size 51 /Prev 101600                /ID [<313a74ebf1d1b077f9086d233cd9cd1e><dff6546817b6c4214ca5d85d9e4be8a5>] >>\nstream\nx\x9ccbd`\xe0g`b``8\t"\x99\xbb\xc1\xec,\x10\xc9\xb5\x06D2\xf6\x82Ef\x83H\xce\x0f 2\xc2\tD\x9a\xd7\x83H\xed\\ \xc9\xe8g\x00b\x87\xbb\x80M\x00\xebbY\x8b \x99W\x83\xd9@\x92\xf1\x7fE9\xd84\x06\xc6\xa1B\x02\x00\x8f\x98\r\x86\nendstream\nendobj\n                                                                                                                                                                                                                                    

In [4]:
from pypdf import PdfReader

reader = PdfReader("Resume_UFS.pdf")

text = ""
for page in reader.pages:
    if page.extract_text():
        text += page.extract_text() + "\n"

print(text[:1000])


Umar Faruk Sarkar
Email: ufs2k19@gmail.com Phone: +91-7364055111
Kolkata, West Bengal
LinkedIn: linkedin.com/in/umar-faruk-sarkar-7883b4193 GitHub: github.com/itsumar1923
Professional Summary
Engineer with strong analytical skills, experienced in data science, software development, and prob-
lem solving. Adept at delivering data-driven insights and building reliable systems.
Technical Skills
Languages: C, C++, Python, JavaScript, SQL, HTML, CSS
Libraries/Frameworks: NumPy, Pandas, Matplotlib, Scikit-learn, TensorFlow, Keras
Areas: Machine Learning, Deep Learning, Data Analysis, Statistical Modeling, NLP, Big Data An-
alytics
Tools: Git, Jupyter Notebook, Microsoft Excel
Experience and Activities
Content Developer, Prithibir_Pathshala2023–Present
•Authored STEM and digital literacy materials for rural learners.
•Collaborated with volunteers to improve curriculum structure.
Technical Writer (Medium)2022–Present
•Published technical articles on data science, algorithms, and tooling.
Learn

In [5]:
!pip install langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

chunks = splitter.split_text(text)

print("Total chunks:", len(chunks))
print(chunks[0][:300])

Total chunks: 5
Umar Faruk Sarkar
Email: ufs2k19@gmail.com Phone: +91-7364055111
Kolkata, West Bengal
LinkedIn: linkedin.com/in/umar-faruk-sarkar-7883b4193 GitHub: github.com/itsumar1923
Professional Summary
Engineer with strong analytical skills, experienced in data science, software development, and prob-
lem sol


In [6]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode(chunks)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
!pip install faiss-cpu
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Vectors stored:", index.ntotal)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 25.1 MB/s  0:00:01
Vectors stored: 5


In [8]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)


/tmp/ipython-input-3896442753.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128255)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm):

In [9]:
def retrieve_context(query, k=3):
    query_embedding = embedder.encode([query])
    distances, indices = index.search(query_embedding, k)

    retrieved_chunks = [chunks[i] for i in indices[0]]
    return "\n\n".join(retrieved_chunks)


In [10]:
def ask_pdf(question):
    context = retrieve_context(question)

    prompt = f"""You are answering strictly from the given document.

### Document:
{context}

### Question:
{question}

### Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.3,
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [11]:
print(ask_pdf("What is this document about?"))


You are answering strictly from the given document.

### Document:
•Authored STEM and digital literacy materials for rural learners.
•Collaborated with volunteers to improve curriculum structure.
Technical Writer (Medium)2022–Present
•Published technical articles on data science, algorithms, and tooling.
Learner – DSMP 2.0, CampusXCurrent
•Advancing skills in Python, statistics, ML pipelines, and deployment.
Projects
Sentiment Analysis of Twitter DataNLP, ML
•Analyzed COVID-19 related tweets using Natural Language Processing and machine learning mod-

Umar Faruk Sarkar
Email: ufs2k19@gmail.com Phone: +91-7364055111
Kolkata, West Bengal
LinkedIn: linkedin.com/in/umar-faruk-sarkar-7883b4193 GitHub: github.com/itsumar1923
Professional Summary
Engineer with strong analytical skills, experienced in data science, software development, and prob-
lem solving. Adept at delivering data-driven insights and building reliable systems.
Technical Skills
Languages: C, C++, Python, JavaScript, SQL, HTM

In [13]:
print(ask_pdf("What is umar's mail id"))


You are answering strictly from the given document.

### Document:
Umar Faruk Sarkar
Email: ufs2k19@gmail.com Phone: +91-7364055111
Kolkata, West Bengal
LinkedIn: linkedin.com/in/umar-faruk-sarkar-7883b4193 GitHub: github.com/itsumar1923
Professional Summary
Engineer with strong analytical skills, experienced in data science, software development, and prob-
lem solving. Adept at delivering data-driven insights and building reliable systems.
Technical Skills
Languages: C, C++, Python, JavaScript, SQL, HTML, CSS

Text Compressor using Huffman EncodingAlgorithms, Systems
•Created a text compression tool achieving significant file size reduction while maintaining integrity.
Live Demo.
Education
M.E. in Information Technology, Jadavpur University (CGPA: 8.01)2024–2026 (Exp.)
B.E. in Electronics & Telecommunication, Jadavpur University (CGPA: 8.43)2019–2023
Certifications and Achievements
•Certifications in Machine Learning and Data Science; active in hackathons.
•WBJEE (2019): Rank642.

•Au